# Signal Upload Workflow

This notebook rebuilds the SHM signal upload workflow as an explicit dry-run linked directly to the public `owi.metadatabase.shm` package architecture.

**What this notebook covers**

- **Import** the SHM API, processing, upload, repository, service, registry, and serializer components used in a realistic signal upload flow.
- **Configure** the Norther packaged signal configs, token source, target turbines, and dry-run API root.
- **Construct** a `ShmAPI`-compatible dry-run signal backend and a lookup service that supplies upload context.
- **Process** raw Norther configuration files through `DefaultSignalConfigProcessor`.
- **Upload** main signals, histories, calibrations, derived signals, and derived calibrations through `ShmSignalUploader.upload_from_processor_files(...)`.
- **Retrieve** the persisted dry-run rows back through `ApiShmRepository` and `SignalService`.

**How to use it**

- Run the notebook top to bottom.
- Keep `TARGET_TURBINES` small when you want a fast dry-run.
- Use the final round-trip cells to verify exactly what the typed SHM layer persisted.


## Mermaid Color Legend

The workflow diagrams in this notebook use a consistent visual language.

- <span style="color:#2B6F77;"><strong>Blue nodes</strong></span>: API calls, processor steps, or service interactions.
- <span style="color:#5E8C61;"><strong>Green nodes</strong></span>: typed output records, upload ids, or persisted dry-run rows.
- <span style="color:#C08B3E;"><strong>Amber nodes</strong></span>: transformations, serializer previews, or lookup preparation.
- <span style="color:#C47A2C;"><strong>Orange decision/request nodes</strong></span>: runtime configuration or upload stages.
- <span style="color:#365A73;"><strong>Blue-grey connectors</strong></span>: data flow between processing, upload, and retrieval.

*Read the diagrams from top to bottom to follow the path from packaged config files to typed SHM records.*


## Imports

These imports assemble the public SHM surfaces needed for a realistic signal dry-run.

**This step prepares**

- **Environment setup** with `Path`, `pandas`, and notebook display helpers.
- **Shared token loading** through the core SDK utility.
- **Signal processing** through `DefaultSignalConfigProcessor`.
- **Transport, upload, and retrieval** through `ShmAPI`, `ShmSignalUploader`, `ApiShmRepository`, and `SignalService`.
- **Typed metadata** through `default_registry`, `DEFAULT_SERIALIZERS`, and `SignalUploadContext`.

*Outcome:* every later stage can focus on signal workflow behavior instead of setup boilerplate.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from owi.metadatabase._utils.utils import load_token_from_env_file

from owi.metadatabase.shm import (
    DEFAULT_SERIALIZERS,
    ApiShmRepository,
    DefaultSignalConfigProcessor,
    ShmAPI,
    ShmEntityName,
    ShmEntityService,
    ShmQuery,
    ShmSignalUploader,
    SignalService,
    default_registry,
)
from owi.metadatabase.shm.upload_context import SignalUploadContext


## Constants

This section defines the runtime configuration for the signal dry-run.

**Review these values before execution**

- **`WORKSPACE_ROOT`**: repository root resolved from the `scripts/` folder.
- **`DEFAULT_ENV_FILE`** and **`TOKEN`**: live-style authentication inputs loaded for parity with real runs.
- **`CONFIGS_ROOT`**, **`SIGNAL_SENSOR_MAP_PATH`**, and **`SENSOR_TC_MAP_PATH`**: packaged Norther inputs consumed by the processor and uploader.
- **`TARGET_TURBINES`**, **`PROJECTSITE`**, and **`PERMISSION_GROUP_IDS`**: the scoped dry-run upload context.

*Outcome:* the notebook exposes every environment-specific input up front, exactly like the Results notebook structure.*


In [3]:
WORKSPACE_ROOT = Path.cwd().resolve().parent
DEFAULT_ENV_FILE = WORKSPACE_ROOT / ".env"
TOKEN = load_token_from_env_file(DEFAULT_ENV_FILE)
BASE_URL = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
CONFIGS_ROOT = WORKSPACE_ROOT / "scripts" / "data" / "Norther" / "signals" / "configs"
SIGNAL_SENSOR_MAP_PATH = WORKSPACE_ROOT / "scripts" / "data" / "Norther" / "signal_sensor_map.json"
SENSOR_TC_MAP_PATH = WORKSPACE_ROOT / "scripts" / "data" / "Norther" / "sensors" / "sensor_temperature_compensations.json"
TARGET_TURBINES = ["NRTC01"]
PROJECTSITE = "Norther"
PERMISSION_GROUP_IDS = [1, 2]

config_summary = pd.DataFrame(
    [
        {
            "workspace_root": str(WORKSPACE_ROOT),
            "configs_root": str(CONFIGS_ROOT),
            "signal_sensor_map": str(SIGNAL_SENSOR_MAP_PATH),
            "sensor_tc_map": str(SENSOR_TC_MAP_PATH),
            "api_root": BASE_URL,
            "token_available": TOKEN is not None,
            "projectsite": PROJECTSITE,
            "target_turbines": ", ".join(TARGET_TURBINES),
        }
    ]
).T.reset_index().rename(columns={"index": "setting", 0: "value"}).set_index("setting")

display(config_summary)


,value
setting,
workspace_root,/home/pietro.dantuono@24SEA.local/Projects/SMA...
configs_root,/home/pietro.dantuono@24SEA.local/Projects/SMA...
signal_sensor_map,/home/pietro.dantuono@24SEA.local/Projects/SMA...
sensor_tc_map,/home/pietro.dantuono@24SEA.local/Projects/SMA...
api_root,https://owimetadatabase-dev.azurewebsites.net/...
token_available,True
projectsite,Norther
target_turbines,NRTC01


## Processor, Lookup, and Dry-Run Transport Setup

This notebook needs more than a transport layer because signal upload also depends on upload context and packaged processor output.

**What happens here**

- Build a deterministic lookup service that returns a `SignalUploadContext`.
- Subclass `ShmAPI` with in-memory persistence for the signal-domain resource set.
- Keep the public `get_*`, `list_*`, `create_*`, and `patch_*` SHM methods intact.
- Use the same repository, registry, and service surfaces that a live backend-backed workflow would use.

*Why this matters:* the dry-run still exercises the real package seams used by `ShmSignalUploader`.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["Constants<br/>paths, token, projectsite"] --> B["DefaultSignalConfigProcessor"]
    A --> C["DemoLookupService<br/>SignalUploadContext"]
    A --> D["DryRunSignalApi<br/>subclasses ShmAPI"]
    D --> E["ApiShmRepository"]
    E --> F["ShmEntityService"]
    F --> G["SignalService"]
    B --> H["ShmSignalUploader"]
    C --> H
    D --> H
    I["default_registry + DEFAULT_SERIALIZERS"] --> F
    G --> J["Typed signal-domain records"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class B,C,D,E,F,G,H api;
    class A,I transform;
    class J output;
```


In [4]:
class DemoLookupService:
    """Deterministic lookup service used by ShmSignalUploader during dry runs."""

    def get_signal_upload_context(self, *, projectsite, assetlocation, permission_group_ids):
        return SignalUploadContext(
            site_id=10,
            asset_location_id=20,
            model_definition_id=50,
            permission_group_ids=permission_group_ids,
            subassembly_ids_by_type={"NAC": 30, "TP": 40, "TW": 41, "MP": 42},
        )


class DryRunSignalApi(ShmAPI):
    """In-memory signal-domain transport that preserves the public ShmAPI surface."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._next_id = 100
        self._signals: list[dict[str, object]] = []
        self._signal_history: list[dict[str, object]] = []
        self._signal_calibrations: list[dict[str, object]] = []
        self._derived_signals: list[dict[str, object]] = []
        self._derived_signal_history: list[dict[str, object]] = []
        self._derived_signal_calibrations: list[dict[str, object]] = []
        self._sensor_type_lookup: dict[tuple[tuple[str, object], ...], int] = {}
        self._sensor_lookup: dict[tuple[tuple[str, object], ...], int] = {}
        self._external_signal_lookup: dict[str, int] = {}

    def _allocate_id(self) -> int:
        self._next_id += 1
        return self._next_id

    def _result(self, rows: list[dict[str, object]], object_id: int | None = None) -> dict[str, object]:
        frame = pd.DataFrame(rows)
        result_id = object_id
        if result_id is None and not frame.empty and "id" in frame.columns:
            result_id = int(frame.iloc[0]["id"])
        return {
            "data": frame,
            "exists": not frame.empty,
            "id": result_id,
            "response": None,
        }

    def _filter(self, rows: list[dict[str, object]], filters: dict[str, object]) -> list[dict[str, object]]:
        return [
            row
            for row in rows
            if all(row.get(key) == value for key, value in filters.items())
        ]

    def _remember(self, store: dict, key) -> int:
        if key not in store:
            store[key] = self._allocate_id()
        return store[key]

    def get_sensor_type(self, **kwargs):
        key = tuple(sorted(kwargs.items()))
        result_id = self._remember(self._sensor_type_lookup, key)
        return self._result([{"id": result_id, **kwargs}], result_id)

    def get_sensor(self, **kwargs):
        normalized = dict(kwargs)
        if normalized.get("serial_number") is not None:
            normalized["serial_number"] = str(normalized["serial_number"])
        if normalized.get("cabinet") is not None:
            normalized["cabinet"] = str(normalized["cabinet"])
        key = tuple(sorted(normalized.items()))
        result_id = self._remember(self._sensor_lookup, key)
        return self._result([{"id": result_id, **normalized}], result_id)

    def get_signal(self, signal_id, **kwargs):
        matches = [row for row in self._signals if row.get("signal_id") == signal_id]
        if matches:
            return self._result(matches[:1], int(matches[0]["id"]))
        result_id = self._remember(self._external_signal_lookup, signal_id)
        return self._result([{"id": result_id, "signal_id": signal_id}], result_id)

    def create_signal(self, payload):
        row = dict(payload)
        row["id"] = self._allocate_id()
        self._signals.append(row)
        return self._result([row])

    def list_signals(self, **kwargs):
        return self._result(self._filter(self._signals, kwargs))

    def get_signal_history(self, **kwargs):
        rows = self._filter(self._signal_history, kwargs)
        return self._result(rows[:1], int(rows[0]["id"]) if rows else None)

    def create_signal_history(self, payload):
        row = dict(payload)
        row["id"] = self._allocate_id()
        self._signal_history.append(row)
        return self._result([row])

    def list_signal_history(self, **kwargs):
        return self._result(self._filter(self._signal_history, kwargs))

    def get_signal_calibration(self, **kwargs):
        rows = self._filter(self._signal_calibrations, kwargs)
        return self._result(rows[:1], int(rows[0]["id"]) if rows else None)

    def create_signal_calibration(self, payload):
        row = dict(payload)
        row["id"] = self._allocate_id()
        self._signal_calibrations.append(row)
        return self._result([row])

    def list_signal_calibrations(self, **kwargs):
        return self._result(self._filter(self._signal_calibrations, kwargs))

    def get_derived_signal(self, **kwargs):
        rows = self._filter(self._derived_signals, kwargs)
        return self._result(rows[:1], int(rows[0]["id"]) if rows else None)

    def create_derived_signal(self, payload):
        row = dict(payload)
        row["id"] = self._allocate_id()
        self._derived_signals.append(row)
        return self._result([row])

    def list_derived_signals(self, **kwargs):
        return self._result(self._filter(self._derived_signals, kwargs))

    def get_derived_signal_history(self, **kwargs):
        rows = self._filter(self._derived_signal_history, kwargs)
        return self._result(rows[:1], int(rows[0]["id"]) if rows else None)

    def create_derived_signal_history(self, payload):
        row = dict(payload)
        row["id"] = self._allocate_id()
        self._derived_signal_history.append(row)
        return self._result([row])

    def list_derived_signal_history(self, **kwargs):
        return self._result(self._filter(self._derived_signal_history, kwargs))

    def patch_derived_signal_history(self, history_id, payload):
        for row in self._derived_signal_history:
            if row["id"] == history_id:
                row.update(dict(payload))
                return self._result([row], history_id)
        return self._result([], history_id)

    def get_derived_signal_calibration(self, **kwargs):
        rows = self._filter(self._derived_signal_calibrations, kwargs)
        return self._result(rows[:1], int(rows[0]["id"]) if rows else None)

    def create_derived_signal_calibration(self, payload):
        row = dict(payload)
        row["id"] = self._allocate_id()
        self._derived_signal_calibrations.append(row)
        return self._result([row])

    def list_derived_signal_calibrations(self, **kwargs):
        return self._result(self._filter(self._derived_signal_calibrations, kwargs))


## Process Packaged Configs and Preview Typed Payload Shapes

Before upload, the notebook makes the processor output explicit and shows how the typed SHM serializer sees one main signal and one derived signal.

**What happens here**

- Run a preview processor on the packaged config file set.
- Inspect the signal-domain registry entries that govern endpoint and record selection.
- Build serializer previews for one main signal and one derived signal.
- Summarize the amount of processor output that the uploader will handle.

*Outcome:* the notebook moves from packaged JSON config files to explicit, typed signal payload preparation.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["packaged config JSON"] --> B["DefaultSignalConfigProcessor.signals_process_data"]
    B --> C["signals_data"]
    B --> D["signals_derived_data"]
    E["default_registry"] --> F["signal entity definitions"]
    G["DEFAULT_SERIALIZERS[signal / derived_signal]"] --> H["preview payloads"]
    C --> H
    D --> H
    C --> I["processor summary table"]

    classDef source fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,B,E,G source;
    class C,D,F,H transform;
    class I output;
```


In [5]:
preview_processor = DefaultSignalConfigProcessor(
    path_configs=CONFIGS_ROOT,
    turbines=TARGET_TURBINES,
)
preview_processor.signals_process_data()

first_signal_name, first_signal_data = next(iter(preview_processor.signals_data[TARGET_TURBINES[0]].items()))
first_derived_signal_name, first_derived_signal_data = next(
    iter(preview_processor.signals_derived_data[TARGET_TURBINES[0]].items())
)

signal_registry_summary = pd.DataFrame(
    [
        {
            "entity": entity_name.value,
            "endpoint": default_registry.get(entity_name).endpoint,
            "record_model": default_registry.get(entity_name).record_model.__name__,
        }
        for entity_name in [
            ShmEntityName.SIGNAL,
            ShmEntityName.SIGNAL_HISTORY,
            ShmEntityName.SIGNAL_CALIBRATION,
            ShmEntityName.DERIVED_SIGNAL,
            ShmEntityName.DERIVED_SIGNAL_HISTORY,
            ShmEntityName.DERIVED_SIGNAL_CALIBRATION,
        ]
    ]
)

signal_payload_preview = DEFAULT_SERIALIZERS[ShmEntityName.SIGNAL].to_payload(
    {"signal_id": first_signal_name, **first_signal_data}
)
derived_signal_payload_preview = DEFAULT_SERIALIZERS[ShmEntityName.DERIVED_SIGNAL].to_payload(
    {"derived_signal_id": first_derived_signal_name, **first_derived_signal_data}
)
processor_summary = pd.DataFrame(
    [
        {
            "turbine": TARGET_TURBINES[0],
            "main_signals": len(preview_processor.signals_data[TARGET_TURBINES[0]]),
            "derived_signals": len(preview_processor.signals_derived_data[TARGET_TURBINES[0]]),
            "first_signal": first_signal_name,
            "first_derived_signal": first_derived_signal_name,
        }
    ]
)

display(signal_registry_summary)
display(pd.DataFrame([signal_payload_preview]))
display(pd.DataFrame([derived_signal_payload_preview]))
display(processor_summary)


,entity,endpoint,record_model
0,signal,signal,SignalRecord
1,signal_history,signalhistory,SignalHistoryRecord
2,signal_calibration,signalcalibration,SignalCalibrationRecord
3,derived_signal,derivedsignal,DerivedSignalRecord
4,derived_signal_history,derivedsignalhistory,DerivedSignalHistoryRecord
5,derived_signal_calibration,derivedsignalcalibration,DerivedSignalCalibrationRecord


,signal_id,detrend,group,heading,level,unit_string,status
0,NRT_C01_NAC_ACC_FA,False,iot,0,100,g,"[{'time': '01/01/1972 00:00', 'name': 'Y', 'st..."


,derived_signal_id,data,calibration,parent_signals
0,NRT_C01_TP_ACC_LAT015_FA,{'name': 'acceleration/yaw_transformation'},"[{'time': '01/01/1972 00:00', 'yaw_parameter':...","[NRT_C01_TP_ACC_LAT015_DEG240_X_nr1, NRT_C01_T..."


,turbine,main_signals,derived_signals,first_signal,first_derived_signal
0,NRTC01,36,11,NRT_C01_NAC_ACC_FA,NRT_C01_TP_ACC_LAT015_FA


## Upload Processed Signals Through the SHM Package

This is the main dry-run upload stage.

**Execution logic**

- Build the dry-run API, repository, entity service, and `SignalService`.
- Recreate the processor instance used for the actual upload call.
- Construct `ShmSignalUploader` with the dry-run API and deterministic lookup service.
- Run `upload_from_processor_files(...)` so the package resolves sensor ids, temperature-compensation signal ids, and upload context exactly as it would in a live workflow.

*Outcome:* the dry-run backend contains main signals, histories, calibrations, derived signals, and derived calibrations produced by the real package logic.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
flowchart TD
    A["packaged configs + file maps"] --> B["DefaultSignalConfigProcessor"]
    B --> C["signals_data / signals_derived_data"]
    D["DemoLookupService"] --> E["ShmSignalUploader.upload_from_processor_files"]
    C --> E
    F["DryRunSignalApi"] --> E
    E --> G["create_signal / create_signal_history / create_signal_calibration"]
    E --> H["create_derived_signal / history / calibration"]
    G --> I["Persisted main signal-domain rows"]
    H --> J["Persisted derived signal-domain rows"]

    classDef decision fill:#FDEBD3,stroke:#C47A2C,color:#5A3412;
    classDef action fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef result fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,D decision;
    class B,C,E,F,G,H action;
    class I,J result;
```


In [6]:
processor = DefaultSignalConfigProcessor(
    path_configs=CONFIGS_ROOT,
    turbines=TARGET_TURBINES,
)
demo_lookup_service = DemoLookupService()
dry_run_api = DryRunSignalApi(api_root=BASE_URL, token=TOKEN or "dry-run-token")
repository = ApiShmRepository(api=dry_run_api)
entity_service = ShmEntityService(repository=repository, registry=default_registry)
signal_service = SignalService(entity_service=entity_service)
uploader = ShmSignalUploader(shm_api=dry_run_api, lookup_service=demo_lookup_service)

lookup_preview = demo_lookup_service.get_signal_upload_context(
    projectsite=PROJECTSITE,
    assetlocation=TARGET_TURBINES[0],
    permission_group_ids=PERMISSION_GROUP_IDS,
)

display(
    pd.DataFrame(
        [
            {
                "ping": dry_run_api.ping(),
                "site_id": lookup_preview.site_id,
                "asset_location_id": lookup_preview.asset_location_id,
                "model_definition_id": lookup_preview.model_definition_id,
                "subassemblies": ", ".join(sorted(lookup_preview.subassembly_ids_by_type)),
            }
        ]
    )
)


,ping,site_id,asset_location_id,model_definition_id,subassemblies
0,ok,10,20,50,"MP, NAC, TP, TW"


In [7]:
upload_results = uploader.upload_from_processor_files(
    projectsite=PROJECTSITE,
    processor=processor,
    path_signal_sensor_map=SIGNAL_SENSOR_MAP_PATH,
    path_sensor_tc_map=SENSOR_TC_MAP_PATH,
    permission_group_ids=PERMISSION_GROUP_IDS,
)

upload_result = upload_results[TARGET_TURBINES[0]]
signal_upload_summary = pd.DataFrame(
    [
        {
            "asset_key": upload_result.asset_key,
            "signals": len(upload_result.signal_ids_by_name),
            "derived_signals": len(upload_result.derived_signal_ids_by_name),
            "signal_history_rows": len(dry_run_api._signal_history),
            "signal_calibrations": len(dry_run_api._signal_calibrations),
            "derived_signal_histories": len(dry_run_api._derived_signal_history),
            "derived_signal_calibrations": len(dry_run_api._derived_signal_calibrations),
        }
    ]
).T.reset_index().rename(columns={"index": "metric", 0: "value"}).set_index("metric")

display(signal_upload_summary)


,value
metric,
asset_key,Norther/NRTC01
signals,27
derived_signals,11
signal_history_rows,31
signal_calibrations,28
derived_signal_histories,11
derived_signal_calibrations,43


## Retrieve Typed Signal Records and Verify the Round Trip

The final stage reads the dry-run rows back through the typed service and serializer stack.

**What this step does**

- List persisted main and derived signal-domain rows through `SignalService`.
- Resolve one main signal and one derived signal through `ShmQuery`.
- Pull raw repository rows and deserialize them through `DEFAULT_SERIALIZERS`.
- Assert exact signal-domain counts so the notebook doubles as a regression check.

*Why this matters:* it proves that processor output, upload orchestration, repository access, registry resolution, and typed deserialization are all aligned.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["DryRunSignalApi stored rows"] --> B["ApiShmRepository.list_records / get_record"]
    B --> C["ShmEntityService"]
    C --> D["SignalService.list_* and get_*"]
    B --> E["DEFAULT_SERIALIZERS.from_mapping"]
    D --> F["Typed signal and derived-signal records"]
    E --> F

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,B,C,D api;
    class E transform;
    class F output;
```


In [8]:
signal_records = signal_service.list_signals()
signal_history_records = signal_service.list_signal_history()
signal_calibration_records = signal_service.list_signal_calibrations()
derived_signal_records = signal_service.list_derived_signals()
derived_signal_history_records = signal_service.list_derived_signal_history()
derived_signal_calibration_records = signal_service.list_derived_signal_calibrations()

first_uploaded_signal_name = next(iter(upload_result.signal_ids_by_name))
first_uploaded_derived_signal_name = next(iter(upload_result.derived_signal_ids_by_name))

first_signal_record = signal_service.get_signal(
    ShmQuery(
        entity=ShmEntityName.SIGNAL,
        backend_filters={"signal_id": first_uploaded_signal_name},
    )
)
first_derived_signal_record = signal_service.get_derived_signal(
    ShmQuery(
        entity=ShmEntityName.DERIVED_SIGNAL,
        backend_filters={"derived_signal_id": first_uploaded_derived_signal_name},
    )
)
raw_signal_frame = repository.list_records(ShmEntityName.SIGNAL)
serializer_roundtrip_signal = DEFAULT_SERIALIZERS[ShmEntityName.SIGNAL].from_mapping(
    raw_signal_frame.iloc[0].to_dict()
)

roundtrip_summary = pd.DataFrame(
    [
        {
            "typed_signals": len(signal_records),
            "typed_signal_history": len(signal_history_records),
            "typed_signal_calibrations": len(signal_calibration_records),
            "typed_derived_signals": len(derived_signal_records),
            "typed_derived_signal_history": len(derived_signal_history_records),
            "typed_derived_signal_calibrations": len(derived_signal_calibration_records),
            "first_signal_id": first_signal_record.id if first_signal_record is not None else None,
            "first_derived_signal_id": first_derived_signal_record.id if first_derived_signal_record is not None else None,
            "serializer_roundtrip_signal": serializer_roundtrip_signal.signal_id,
        }
    ]
).T.reset_index().rename(columns={"index": "metric", 0: "value"}).set_index("metric")

display(roundtrip_summary)
display(pd.DataFrame([record.model_dump(mode="json") for record in signal_records[:3]]))
display(pd.DataFrame([record.model_dump(mode="json") for record in derived_signal_records[:3]]))
display(pd.DataFrame([record.model_dump(mode="json") for record in signal_calibration_records[:3]]))


,value
metric,
typed_signals,27
typed_signal_history,31
typed_signal_calibrations,28
typed_derived_signals,11
typed_derived_signal_history,11
typed_derived_signal_calibrations,43
first_signal_id,178
first_derived_signal_id,264
serializer_roundtrip_signal,NRT_C01_NAC_ACC_FA


,id,signal_id,title,subtitle,description,signal_type,status,site,asset_location,asset_location_id,model_definition,sub_assembly,heading,level,orientation,stats,visibility,visibility_groups,data_additional
0,178,NRT_C01_NAC_ACC_FA,None,None,None,ACC,None,10,20,None,50,None,0,100.0,None,None,usergroup,"[1, 2]","{'detrend': False, 'group': 'iot', 'unit_strin..."
1,179,NRT_C01_NAC_ACC_SS,None,None,None,ACC,None,10,20,None,50,None,0,100.0,None,None,usergroup,"[1, 2]","{'detrend': False, 'group': 'iot', 'unit_strin..."
2,180,NRT_C01_NAC_ACC_Z,None,None,None,ACC,None,10,20,None,50,None,0,100.0,None,None,usergroup,"[1, 2]","{'group': 'iot', 'unit_string': 'g'}"


,id,derived_signal_id,title,subtitle,description,signal_type,status,site,asset_location,asset_location_id,model_definition,visibility,visibility_groups,data_additional
0,264,NRT_C01_TP_ACC_LAT015_FA,None,None,None,ACC,None,10,20,None,50,usergroup,"[1, 2]",{'name': 'acceleration/yaw_transformation'}
1,265,NRT_C01_TP_ACC_LAT015_SS,None,None,None,ACC,None,10,20,None,50,usergroup,"[1, 2]",{'name': 'acceleration/yaw_transformation'}
2,266,NRT_C01_TW_ACC_LAT069_FA,None,None,None,ACC,None,10,20,None,50,usergroup,"[1, 2]",{'name': 'acceleration/yaw_transformation'}


,id,signal_id,calibration_date,tempcomp_signal_id,status_approval,data
0,211,183,1972-01-01T00:00:00,NaN,yes,{'offset': 6.77357}
1,213,184,1972-01-01T00:00:00,NaN,yes,{'offset': 4.05996}
2,216,185,1972-01-01T00:00:00,166.0,yes,"{'offset': -603.19703, 'Coefficients': [0, 0.0..."


In [9]:
assert signal_upload_summary.loc["signals", "value"] == 27
assert signal_upload_summary.loc["derived_signals", "value"] == 11
assert signal_upload_summary.loc["signal_history_rows", "value"] == 31
assert signal_upload_summary.loc["signal_calibrations", "value"] == 28
assert signal_upload_summary.loc["derived_signal_histories", "value"] == 11
assert signal_upload_summary.loc["derived_signal_calibrations", "value"] == 43
assert len(signal_records) == 27
assert len(signal_history_records) == 31
assert len(signal_calibration_records) == 28
assert len(derived_signal_records) == 11
assert len(derived_signal_history_records) == 11
assert len(derived_signal_calibration_records) == 43
assert first_signal_record is not None and first_signal_record.signal_id == first_uploaded_signal_name
assert first_derived_signal_record is not None and first_derived_signal_record.derived_signal_id == first_uploaded_derived_signal_name

pd.DataFrame(
    [
        {
            "status": "ok",
            "signal_count": len(signal_records),
            "derived_signal_count": len(derived_signal_records),
            "signal_history_count": len(signal_history_records),
            "signal_calibration_count": len(signal_calibration_records),
        }
    ]
)


,status,signal_count,derived_signal_count,signal_history_count,signal_calibration_count
0,ok,27,11,31,28
